# PCCP census: AI classification, agreement and modification taxonomy

Each full submission or PMA supplement identifier represents one authorization record through December 31, 2025. AI-status agreement uses the original SB/OF ratings; final classifications use their agreement or MO adjudication. Modification categories use the supplied SB/OF/MO/Final_verdict columns.

Run all cells in order from a clean kernel. The notebook validates the supplied classifications, calculates agreement and descriptive results, and writes four CSVs under `outputs/`. Plots display inline.


## Inputs and coding contract

| File in data/census/ | Required content |
| --- | --- |
| ai_status_review.csv | All eligible identifiers and names; original SB/OF AI or Non-AI ratings; MO on disagreements only. |
| eligible_metadata.csv | The same eligible identifiers, dates, pathway, specialty, panel, product code and FDA links. |
| ai_devices_taxonomy.csv | Exactly the final AI cohort; Submission_Number, Device_Name, Final_AI_Status, SB, OF, MO, Final_verdict. Additional source/review columns are ignored. |

Category sets contain unique semicolon-separated codes 1–9, such as `1;4`.
Order has no scientific meaning. `insufficient detail` is a distinct verdict; blank
means unresolved. On agreements, Final_verdict must equal the shared reviewer verdict
and MO must be blank. On disagreements, MO contains the **complete reconciled set**,
and Final_verdict must match it.
Shared categories are retained; only disputed categories can be included or omitted.

Category frequencies use all final AI authorizations as denominator and can overlap.
Unclassifiable records are reported separately. Category-count distributions describe
assigned categories per authorization, not numbers of individual changes, complexity or risk.
Taxonomy agreement compares complete original verdicts, with an additional result for
classifiable pairs; unclassifiable records never become all-negative binary ratings.

In [ ]:
from pathlib import Path
import hashlib
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import MaxNLocator
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT_DIR = ROOT / "data/census"
REVIEW_PATH = INPUT_DIR / "ai_status_review.csv"
SOURCE_PATH = INPUT_DIR / "eligible_metadata.csv"
TAXONOMY_PATH = INPUT_DIR / "ai_devices_taxonomy.csv"
OUTPUT_DIR = ROOT / "outputs"
CUTOFF = pd.Timestamp("2025-12-31")
FIRST_YEAR = 2020
KEY = "Submission_Number"
STATUSES = ["AI", "Non-AI"]
INSUFFICIENT = "insufficient detail"
CATEGORIES = {
    "1": "Model and algorithm changes", "2": "Patient-population changes",
    "3": "Clinical-function changes", "4": "Input and compatibility changes",
    "5": "Software infrastructure changes", "6": "User-interface changes",
    "7": "Labeling changes", "8": "Hardware changes", "9": "Cleaning changes",
}

# Scientific Figure Style Guide v1.1; no vault dependency at runtime.
FIGURE_STYLE_VERSION = "1.1"
FIGURE_FONT = "Arial"  # Automatically falls back to DejaVu Sans if unavailable.
FIGURE_WIDTH = 7.2  # Working inches, not final publication dimensions.
VERTICAL_CATEGORY_LIMIT = 5
COLORS = {"teal": "#5F828E", "dark_teal": "#2E5661", "orange": "#E6A052",
          "warm_gray": "#B5AE9B", "light_blue": "#ADC6CF"}

def parse_categories(value, field, identifier):
    if not isinstance(value, str) or not value or value != value.strip():
        raise ValueError(f"{identifier}: {field} is blank or has surrounding whitespace.")
    if value == INSUFFICIENT:
        return INSUFFICIENT
    codes = value.split(";")
    if len(codes) != len(set(codes)) or not set(codes).issubset(CATEGORIES):
        raise ValueError(f"{identifier}: invalid or duplicate category in {field}: {value!r}")
    return frozenset(codes)

def canonical_categories(value):
    return value if value == INSUFFICIENT else ";".join(sorted(value, key=int))

def validate_final_taxonomy(frame):
    missing_final = frame.loc[frame["Final_verdict"].eq(""), KEY].tolist()
    if missing_final:
        raise ValueError("Missing final taxonomy verdicts for "
                         + ", ".join(missing_final))
    parsed = []
    for row in frame.to_dict("records"):
        identifier = row[KEY]
        sb = parse_categories(row["SB"], "SB", identifier)
        of = parse_categories(row["OF"], "OF", identifier)
        final = parse_categories(row["Final_verdict"], "Final_verdict", identifier)
        if sb == of:
            if row["MO"] or final != sb:
                raise ValueError(f"{identifier}: agreed categories must carry forward unchanged; MO stays blank.")
        else:
            mo = parse_categories(row["MO"], "MO", identifier)
            if final != mo:
                raise ValueError(f"{identifier}: final verdict differs from MO adjudication.")
            if not all(isinstance(x, frozenset) for x in [sb, of, final]):
                raise ValueError(f"{identifier}: unclassifiable disagreement requires separate author review.")
            if not (sb & of).issubset(final) or not final.issubset(sb | of):
                raise ValueError(f"{identifier}: adjudication must preserve shared categories and decide only disputed ones.")
        parsed.append((sb, of, final))
    return parsed

## Validate inputs

Check schema, unique identifiers, eligible/AI cohort coverage, device names, decision dates and consistency of the final classifications. The supplied source table contains the final eligible cohort. Missing or inconsistent classifications stop the calculation.


In [ ]:
input_paths = [REVIEW_PATH, SOURCE_PATH, TAXONOMY_PATH]
missing = [str(p.relative_to(ROOT)) for p in input_paths if not p.is_file()]
if missing:
    raise FileNotFoundError("Required input unavailable: " + ", ".join(missing))
review = pd.read_csv(REVIEW_PATH, dtype=str, keep_default_na=False)
source = pd.read_csv(SOURCE_PATH, dtype=str, keep_default_na=False)
taxonomy = pd.read_csv(TAXONOMY_PATH, dtype=str, keep_default_na=False)
required_review = [KEY, "Device_Name", "SB", "OF", "MO"]
required_source = [KEY, "Device_Name", "Submission_Type", "Decision_Date",
                   "Medical_Specialty", "Review_Panel", "Product_Code",
                   "fda_record_url", "source_document_url"]
required_taxonomy = [KEY, "Device_Name", "Final_AI_Status", "SB", "OF", "MO", "Final_verdict"]
for name, frame, required in [("Review", review, required_review),
                               ("Source", source, required_source),
                               ("Taxonomy", taxonomy, required_taxonomy)]:
    absent = set(required) - set(frame.columns)
    if absent:
        raise ValueError(f"{name}: missing columns {sorted(absent)}")
    if frame.empty or frame[KEY].str.strip().eq("").any() or frame[KEY].duplicated().any():
        raise ValueError(f"{name}: require unique, nonblank identifiers and nonempty data.")
    if frame[KEY].ne(frame[KEY].str.strip()).any():
        raise ValueError(f"{name}: correct identifier whitespace in the source.")

# Validate the final classifications before calculating results.
parsed_taxonomy = validate_final_taxonomy(taxonomy)
if set(review[KEY]) != set(source[KEY]):
    raise ValueError("Review and eligible source identifiers must match exactly.")
if not review[["SB", "OF"]].isin(STATUSES).all().all():
    raise ValueError("SB and OF require complete AI/Non-AI ratings.")
disagrees = review["SB"].ne(review["OF"])
if not review.loc[disagrees, "MO"].isin(STATUSES).all():
    raise ValueError("Every AI-status disagreement requires an MO decision.")
if review.loc[~disagrees, "MO"].ne("").any():
    raise ValueError("MO must remain blank on agreed AI-status records.")
final_verdicts = review[required_review].copy()
final_verdicts["Final_AI_Status"] = review["SB"].where(~disagrees, review["MO"])
final_verdicts["Final_Decision_Source"] = np.where(disagrees, "MO adjudication", "SB/OF agreement")
census = source.merge(final_verdicts, on=KEY, validate="one_to_one", suffixes=("", "_review"))
if census["Device_Name"].ne(census["Device_Name_review"]).any():
    raise ValueError("Review/source device names differ.")
census = census.drop(columns="Device_Name_review")
if "eligibility_status" in source and not source["eligibility_status"].eq("included").all():
    raise ValueError("Source contains a record not marked included.")
if "authorization_assessment" in source and not source["authorization_assessment"].eq("supported").all():
    raise ValueError("Source contains an unresolved authorization assessment.")
census["Decision_Date_parsed"] = pd.to_datetime(census["Decision_Date"], format="%Y-%m-%d", errors="raise")
if census["Decision_Date_parsed"].isna().any() or census["Decision_Date_parsed"].gt(CUTOFF).any():
    raise ValueError("Decision dates must be complete and within the cutoff.")
census["Decision_Year"] = census["Decision_Date_parsed"].dt.year
if census["Decision_Year"].lt(FIRST_YEAR).any():
    raise ValueError("An authorization precedes FIRST_YEAR.")
ai = census.loc[census["Final_AI_Status"].eq("AI")].copy()
if ai.empty or set(taxonomy[KEY]) != set(ai[KEY]) or not taxonomy["Final_AI_Status"].eq("AI").all():
    raise ValueError("Taxonomy must cover exactly the final AI cohort.")
if not taxonomy.set_index(KEY)["Device_Name"].equals(ai.set_index(KEY)["Device_Name"].reindex(taxonomy[KEY])):
    raise ValueError("Taxonomy/source device names differ.")
taxonomy = taxonomy.copy()
taxonomy["SB_set"] = [x[0] for x in parsed_taxonomy]
taxonomy["OF_set"] = [x[1] for x in parsed_taxonomy]
taxonomy["Final_set"] = [x[2] for x in parsed_taxonomy]

# The shared signature ties exported tables to the exact three input files.
input_hashes = [(p.name, hashlib.sha256(p.read_bytes()).hexdigest()) for p in input_paths]
input_signature = hashlib.sha256("\n".join(f"{name}:{sha}" for name, sha in input_hashes).encode()).hexdigest()
display(pd.DataFrame(input_hashes, columns=["input", "sha256"]))
print(f"Validated {len(census)} eligible authorizations and {len(ai)} AI taxonomy records.")

## Inter-rater agreement before adjudication

AI-status agreement uses all original SB/OF binary pairs. Let a denote both AI,
b SB AI/OF Non-AI, c SB Non-AI/OF AI and d both Non-AI. Exact agreement is (a+d)/N;
positive/negative agreement are 2a/(2a+b+c) and 2d/(2d+b+c). Cohen's expected
agreement is [(a+b)(a+c)+(c+d)(b+d)]/N². Binary AC1 uses p=(2a+b+c)/(2N)
and expected agreement 2p(1−p). Undefined denominators remain missing.

Taxonomy agreement compares original complete SB/OF verdicts without regard to code
order. Report exact agreement for all verdict pairs (including agreement on insufficient
detail) and separately where both reviewers supplied category sets.

For each of the nine modification categories, treat presence/absence as a separate binary
rating and calculate exact, positive and negative agreement, Cohen's kappa and Gwet's AC1
using the same formulas as AI status. Include only authorizations where both original
reviewers supplied category sets. Exclude insufficient-detail verdicts from these binary
calculations rather than treating them as nine absent categories. Show the four contingency
counts, paired denominator and excluded count for each category. Undefined coefficients
remain missing. Retain complete-set agreement separately; no pooled coefficient,
acceptance threshold or post-adjudication reliability is introduced.

In [ ]:
agreement_table = pd.crosstab(review["SB"], review["OF"]).reindex(index=STATUSES, columns=STATUSES, fill_value=0)
a, b = int(agreement_table.loc["AI", "AI"]), int(agreement_table.loc["AI", "Non-AI"])
c, d = int(agreement_table.loc["Non-AI", "AI"]), int(agreement_table.loc["Non-AI", "Non-AI"])
n = a + b + c + d
po = (a + d) / n
pe_kappa = ((a+b)*(a+c) + (c+d)*(b+d)) / n**2
p = (2*a+b+c) / (2*n)
pe_ac1 = 2*p*(1-p)

def ratio(numerator, denominator):
    return numerator / denominator if denominator != 0 else np.nan

agreement_counts = pd.DataFrame([
    {"SB": sb, "OF": of, "n": int(agreement_table.loc[sb, of])}
    for sb in STATUSES for of in STATUSES
])
irr = pd.DataFrame([
    {"metric": "Exact agreement", "value": po, "numerator": a+d, "denominator": n},
    {"metric": "AI-specific positive agreement", "value": ratio(2*a,2*a+b+c), "numerator": 2*a, "denominator": 2*a+b+c},
    {"metric": "Non-AI-specific negative agreement", "value": ratio(2*d,2*d+b+c), "numerator": 2*d, "denominator": 2*d+b+c},
    {"metric": "Cohen kappa", "value": ratio(po-pe_kappa,1-pe_kappa), "numerator": po-pe_kappa, "denominator": 1-pe_kappa},
    {"metric": "Gwet AC1", "value": ratio(po-pe_ac1,1-pe_ac1), "numerator": po-pe_ac1, "denominator": 1-pe_ac1},
])
irr["n_pairs"] = n
display(agreement_table)
display(irr)


agreement_counts.insert(0, "assessment", "AI status")
irr.insert(0, "assessment", "AI status")
taxonomy_pairs = pd.DataFrame({"SB": taxonomy.SB_set.map(canonical_categories),
                               "OF": taxonomy.OF_set.map(canonical_categories)})
taxonomy_contingency = taxonomy_pairs.groupby(["SB", "OF"], sort=True).size().reset_index(name="n")
taxonomy_contingency.insert(0, "assessment", "Taxonomy verdict")
agreement_counts = pd.concat([agreement_counts, taxonomy_contingency], ignore_index=True)
classifiable_pairs = taxonomy_pairs.SB.ne(INSUFFICIENT) & taxonomy_pairs.OF.ne(INSUFFICIENT)
taxonomy_agreement = []
for label, subset in [("Taxonomy all verdicts", taxonomy_pairs),
                       ("Taxonomy classifiable pairs", taxonomy_pairs.loc[classifiable_pairs])]:
    matched = int(subset.SB.eq(subset.OF).sum())
    taxonomy_agreement.append({"assessment": label, "metric": "Exact agreement",
                               "value": ratio(matched, len(subset)), "numerator": matched,
                               "denominator": len(subset), "n_pairs": len(subset)})
irr = pd.concat([irr, pd.DataFrame(taxonomy_agreement)], ignore_index=True)
display(taxonomy_contingency)
display(pd.DataFrame(taxonomy_agreement))

# Category-specific reliability uses original ratings only, never Final_set or MO.
category_count_rows = []
category_irr_rows = []
category_summary_rows = []
paired_taxonomy = taxonomy.loc[classifiable_pairs]
category_n = len(paired_taxonomy)
excluded_category_n = len(taxonomy) - category_n
for code, label in CATEGORIES.items():
    sb_present = paired_taxonomy.SB_set.map(lambda codes: code in codes)
    of_present = paired_taxonomy.OF_set.map(lambda codes: code in codes)
    a = int((sb_present & of_present).sum())
    b = int((sb_present & ~of_present).sum())
    c = int((~sb_present & of_present).sum())
    d = int((~sb_present & ~of_present).sum())
    if a + b + c + d != category_n:
        raise ValueError(f"Category {code}: paired counts do not reconcile.")
    context = {"assessment": "Taxonomy category", "category_code": code,
               "category": label, "excluded_n": excluded_category_n}
    for sb, of, count in [("Present", "Present", a), ("Present", "Absent", b),
                           ("Absent", "Present", c), ("Absent", "Absent", d)]:
        category_count_rows.append({**context, "SB": sb, "OF": of, "n": count})
    observed = ratio(a + d, category_n)
    chance_kappa = ratio((a+b)*(a+c) + (c+d)*(b+d), category_n**2)
    mean_presence = ratio(2*a+b+c, 2*category_n)
    chance_ac1 = 2 * mean_presence * (1 - mean_presence)
    metrics = [
        ("Exact agreement", a+d, category_n),
        ("Category-specific positive agreement", 2*a, 2*a+b+c),
        ("Category-specific negative agreement", 2*d, 2*d+b+c),
        ("Cohen kappa", observed-chance_kappa, 1-chance_kappa),
        ("Gwet AC1", observed-chance_ac1, 1-chance_ac1),
    ]
    for metric, numerator, denominator in metrics:
        category_irr_rows.append({**context, "metric": metric,
                                  "value": ratio(numerator, denominator),
                                  "numerator": numerator, "denominator": denominator,
                                  "n_pairs": category_n})
    category_summary_rows.append({"category_code": code, "category": label,
                                  "both_present": a, "SB_only": b, "OF_only": c,
                                  "both_absent": d, "n_pairs": category_n,
                                  "excluded_n": excluded_category_n,
                                  "exact_agreement": observed,
                                  "cohens_kappa": ratio(observed-chance_kappa, 1-chance_kappa),
                                  "gwets_ac1": ratio(observed-chance_ac1, 1-chance_ac1)})
agreement_counts = pd.concat([agreement_counts, pd.DataFrame(category_count_rows)], ignore_index=True)
irr = pd.concat([irr, pd.DataFrame(category_irr_rows)], ignore_index=True)
category_reliability = pd.DataFrame(category_summary_rows)
print(f"Category-specific reliability: {category_n} paired classifiable authorizations; "
      f"{excluded_category_n} insufficient-detail pairs excluded.")
display(category_reliability)


## Descriptive census results

Summarize all eligible authorizations and the AI and Non-AI subsets separately. These
are counts and proportions among identified eligible PCCP authorizations, not uptake
rates among all FDA authorizations. Preserve every year from 2020 through the cutoff,
including years with no records. Medical specialty and review panel retain their separate
source meanings. Missing categorical metadata are shown as `Not reported` and remain
in denominators; no specialty is inferred from the product code.

The source table has no manufacturer field, so manufacturer normalization and summaries
require additional source data. They are not inferred from device names.


In [ ]:
cohorts = {"All eligible": census, "AI": ai,
           "Non-AI": census.loc[census["Final_AI_Status"].eq("Non-AI")]}
dimensions = {"Final_AI_Status": "AI status", "Decision_Year": "Decision year",
              "Submission_Type": "Authorization pathway", "Medical_Specialty": "Medical specialty",
              "Review_Panel": "FDA review panel", "Product_Code": "Product code"}
descriptive_rows = []
for column, label in dimensions.items():
    if column == "Decision_Year":
        categories = list(range(FIRST_YEAR, CUTOFF.year+1))
    elif column == "Final_AI_Status":
        categories = STATUSES
    else:
        categories = sorted(census[column].replace("", "Not reported").unique())
    for cohort, frame in cohorts.items():
        if column == "Final_AI_Status" and cohort != "All eligible":
            continue
        values = frame[column] if column == "Decision_Year" else frame[column].replace("", "Not reported")
        counts = values.value_counts().reindex(categories, fill_value=0)
        for category, count in counts.items():
            descriptive_rows.append({"dimension": label, "cohort": cohort, "category": str(category),
                                     "n": int(count), "denominator": len(frame),
                                     "proportion": ratio(int(count), len(frame))})
descriptive_counts = pd.DataFrame(descriptive_rows)
for label in dimensions.values():
    print(label)
    display(descriptive_counts.loc[descriptive_counts["dimension"].eq(label)])


## Final modification taxonomy

Nine category rows include zero counts. Each authorization contributes once to each
applicable category; denominators include the entire final AI cohort. A separate coding
status reports classifiable/insufficient-detail totals. The category-count distribution
uses 1–9 for classifiable records and a separate Insufficient detail row, never zero.
These three views share one aggregate table for Figure 1C/D and supplementary reporting.

In [ ]:
final_sets = taxonomy["Final_set"]
classifiable = final_sets.map(lambda value: isinstance(value, frozenset))
classifiable_n = int(classifiable.sum())
insufficient_n = int((~classifiable).sum())
taxonomy_rows = []
def add_taxonomy_count(dimension, code, label, count):
    taxonomy_rows.append({"dimension": dimension, "category_code": str(code), "category": label,
                           "n": int(count), "denominator": len(ai),
                           "proportion": ratio(int(count), len(ai)),
                           "classifiable_n": classifiable_n, "insufficient_detail_n": insufficient_n})
for code, label in CATEGORIES.items():
    count = sum(code in value for value in final_sets[classifiable])
    add_taxonomy_count("Modification category", code, label, count)
add_taxonomy_count("Coding status", "classifiable", "Classifiable", classifiable_n)
add_taxonomy_count("Coding status", INSUFFICIENT, "Insufficient detail", insufficient_n)
category_lengths = final_sets[classifiable].map(len)
for count in range(1, 10):
    add_taxonomy_count("Category count", count, str(count), int(category_lengths.eq(count).sum()))
add_taxonomy_count("Category count", INSUFFICIENT, "Insufficient detail", insufficient_n)
taxonomy_counts = pd.DataFrame(taxonomy_rows)
if taxonomy_counts.loc[taxonomy_counts.dimension.eq("Category count"), "n"].sum() != len(ai):
    raise ValueError("Category-count distribution does not reconcile to the AI cohort.")
display(taxonomy_counts)

## Figures

House style v1.1: white backgrounds, regular Arial with automatic DejaVu Sans fallback,
11 pt axes and ticks, 10 pt value labels and legends, thin gray axes and grids, flat teal
bars and zero-based count axes.
The resolved font and its hash are displayed below.

Single-series figures use vertical bars for up to five categories and horizontal bars
otherwise. Annual counts use chronological vertical bars even with six years, retaining
the previously accepted census annual-chart layout. AI and Non-AI use stable distinct
colors and legend labels in comparisons. Figure headings and denominators appear in
notebook text, outside the graphics. These are working figures; no image files are exported.


In [ ]:
try:
    font_path = Path(font_manager.findfont(font_manager.FontProperties(family=FIGURE_FONT), fallback_to_default=False))
except ValueError:
    print(f"{FIGURE_FONT} unavailable; using DejaVu Sans.")
    FIGURE_FONT = "DejaVu Sans"
    font_path = Path(font_manager.findfont(font_manager.FontProperties(family=FIGURE_FONT), fallback_to_default=False))
print(f"Style {FIGURE_STYLE_VERSION}; font {FIGURE_FONT}; font file {font_path.name}; "
      f"SHA-256 {hashlib.sha256(font_path.read_bytes()).hexdigest()}")
plt.rcdefaults()
plt.rcParams.update({
    "font.family": FIGURE_FONT, "font.size": 11, "font.weight": "normal",
    "figure.facecolor": "white", "axes.facecolor": "white", "figure.dpi": 100,
    "axes.edgecolor": "#404040", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": True, "axes.spines.bottom": True,
    "axes.grid": False, "axes.axisbelow": True, "grid.color": "#DDDDDD",
    "grid.linewidth": 0.6, "patch.edgecolor": "#404040", "patch.linewidth": 0.6,
    "lines.linewidth": 0.9, "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "legend.fontsize": 10, "legend.fancybox": False,
    "legend.framealpha": 1.0, "legend.edgecolor": "#404040",
})

def count_plot(labels, counts, category_axis):
    horizontal = len(labels) > VERTICAL_CATEGORY_LIMIT
    wrapped = [textwrap.fill(str(label), 32 if horizontal else 18) for label in labels]
    height = max(3.2, 0.34*sum(label.count("\n")+1 for label in wrapped)+0.7) if horizontal else 3.8
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, height), layout="constrained")
    positions = np.arange(len(labels))
    if horizontal:
        bars = ax.barh(positions, counts, height=0.65, color=COLORS["teal"], edgecolor="#404040", linewidth=0.6)
        ax.set_yticks(positions, labels=wrapped)
        ax.invert_yaxis()
        ax.set_xlabel("Authorizations, n")
        ax.set_ylabel(category_axis)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(axis="x")
        ax.set_xlim(0, max(1, max(counts)*1.17))
    else:
        bars = ax.bar(positions, counts, width=0.65, color=COLORS["teal"], edgecolor="#404040", linewidth=0.6)
        ax.set_xticks(positions, labels=wrapped)
        ax.set_ylabel("Authorizations, n")
        ax.set_xlabel(category_axis)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(axis="y")
        ax.set_ylim(0, max(1, max(counts)*1.17))
    ax.bar_label(bars, labels=[str(int(n)) for n in counts], padding=3, fontsize=10)
    plt.show()
    plt.close(fig)

def label_vertical_bars(ax, containers):
    # Measure labels after the square axes and limits have been laid out.
    labels = []
    for bars in containers:
        for bar in bars:
            value = int(bar.get_height())
            if not value:
                continue
            label = ax.text(bar.get_x()+bar.get_width()/2,
                            bar.get_y()+bar.get_height()/2, str(value),
                            ha="center", va="center", fontsize=10)
            labels.append((bar, label))
    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()
    for bar, label in labels:
        bar_box = bar.get_window_extent(renderer)
        text_box = label.get_window_extent(renderer)
        if text_box.height+6 <= bar_box.height and text_box.width+6 <= bar_box.width:
            r, g, b, _ = bar.get_facecolor()
            label.set_color("white" if 0.2126*r+0.7152*g+0.0722*b < 0.6 else "#222222")
        else:
            # A right-side callout avoids overlapping the next stacked segment.
            label.remove()
            ax.annotate(str(int(bar.get_height())),
                        xy=(bar.get_x()+bar.get_width(), bar.get_y()+bar.get_height()/2),
                        xytext=(5, 0), textcoords="offset points", ha="left", va="center",
                        fontsize=10, color="#222222", annotation_clip=False)

def comparison_plot(table, category_axis, stacked=False):
    # Used only for chronological years and the short pathway comparison.
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_WIDTH) if stacked else (FIGURE_WIDTH, 4), layout="constrained")
    if stacked:
        ax.set_box_aspect(1)
    x = np.arange(len(table))
    bottom = np.zeros(len(table))
    containers = []
    for i, status in enumerate(STATUSES):
        bars = ax.bar(x if stacked else x+(i-0.5)*0.325, table[status],
                      width=0.65 if stacked else 0.325, bottom=bottom if stacked else None,
                      color=COLORS["teal"] if status == "AI" else COLORS["orange"],
                      edgecolor="#404040", linewidth=0.6, label=status)
        containers.append(bars)
        if not stacked:
            ax.bar_label(bars, labels=[str(int(v)) for v in table[status]], padding=3, fontsize=10)
        if stacked:
            bottom += table[status].to_numpy()
    ax.set_xticks(x, labels=[textwrap.fill(str(v), 18) for v in table.index])
    ax.set_xlabel(category_axis)
    ax.set_ylabel("Authorizations, n")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylim(0, 120 if stacked else max(1, table.to_numpy().max()*1.22))
    ax.grid(axis="y")
    ax.legend(loc="upper left")
    if stacked:
        label_vertical_bars(ax, containers)
    plt.show()
    plt.close(fig)


### AI status among all eligible PCCP authorizations

The denominator is the full eligible cohort.


In [ ]:
status_counts = census["Final_AI_Status"].value_counts().reindex(STATUSES, fill_value=0)
print(f"Denominator: {len(census)} eligible authorizations")
count_plot(STATUSES, status_counts.to_numpy(), "Final AI status")


### Annual authorizations by final AI status

Each year has one stacked bar with AI and Non-AI segments. Counts retain all calendar years through the study cutoff, including zero-count years.


In [ ]:
annual = pd.crosstab(census["Decision_Year"], census["Final_AI_Status"]).reindex(
    index=range(FIRST_YEAR,CUTOFF.year+1), columns=STATUSES, fill_value=0)
print(f"{len(census)} eligible authorizations through {CUTOFF.date()}")
comparison_plot(annual, "Decision year", stacked=True)


### Authorization pathway by final AI status

PMA supplements retain their full identifiers and remain in the PMA pathway.


In [ ]:
pathways = pd.crosstab(census["Submission_Type"].replace("", "Not reported"),
                       census["Final_AI_Status"]).reindex(columns=STATUSES, fill_value=0)
if len(pathways) <= VERTICAL_CATEGORY_LIMIT:
    comparison_plot(pathways, "Authorization pathway")
else:
    # Preserve the category-count orientation rule if future data add pathways.
    for status in STATUSES:
        print(status)
        count_plot(pathways.index.tolist(), pathways[status].to_numpy(), "Authorization pathway")


### Medical specialty of AI authorizations

Source specialties are used as recorded; missing metadata remain visible.


In [ ]:
specialties = ai["Medical_Specialty"].replace("", "Not reported").value_counts().sort_index()
print(f"Denominator: {len(ai)} AI authorizations")
count_plot(specialties.index.tolist(), specialties.to_numpy(), "Medical specialty")


### Modification categories within the AI cohort

The denominator includes all AI authorizations. Insufficient-detail records are reported
separately and contribute to no named category.

In [ ]:
part = taxonomy_counts.loc[taxonomy_counts.dimension.eq("Modification category")]
print(f"AI authorizations: {len(ai)}; classifiable: {classifiable_n}; insufficient detail: {insufficient_n}")
count_plot(part.category.tolist(), part.n.to_numpy(), "Modification category")

### Number of modification categories per AI authorization

Each bar counts authorization records assigned exactly that many distinct modification
categories in the final adjudicated coding. This measures category breadth, not the
number of individual changes. Each full submission or PMA supplement is one record;
records are not collapsed into unique devices. Insufficient-detail records have unknown
category counts and are reported separately, never as zero. All nine possible counts
remain visible, including unobserved counts.


In [ ]:
category_distribution = taxonomy_counts.loc[
    taxonomy_counts.dimension.eq("Category count")
    & taxonomy_counts.category_code.ne(INSUFFICIENT)
].copy()
category_distribution["category_count"] = category_distribution.category_code.astype(int)
category_distribution = category_distribution.sort_values("category_count")
assert category_distribution.n.sum() == classifiable_n
print(f"Classifiable AI authorizations: {classifiable_n}/{len(ai)}; "
      f"insufficient detail (not plotted): {insufficient_n}/{len(ai)}")
display(category_distribution[["category_count", "n"]].rename(columns={
    "category_count": "Modification categories per authorization", "n": "Authorizations, n"}))
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_WIDTH), layout="constrained")
ax.set_box_aspect(1)
bars = ax.bar(category_distribution.category_count, category_distribution.n,
              width=0.65, color=COLORS["teal"], edgecolor="#404040", linewidth=0.6)
ax.set_xticks(range(1, 10))
ax.set_xlabel("Modification categories per authorization, n")
ax.set_ylabel("AI authorizations, n")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylim(0, max(1, category_distribution.n.max()*1.17))
ax.grid(axis="y")
label_vertical_bars(ax, [bars])
plt.show()
plt.close(fig)


## Save four aggregate result tables

Only this cell writes files, under outputs/. Derived tables can be
regenerated; inputs and reviewer records are read-only. census_agreement_counts.csv and
census_irr.csv distinguish AI status, complete taxonomy verdicts and category-specific
presence/absence agreement. Category rows include their code, label and excluded count.
census_descriptive_counts.csv contains cohort composition and metadata distributions.
census_taxonomy_counts.csv contains category prevalence, assessability and category-count
distributions. All four carry the same input_signature identifying the exact census input files. Proportions are numeric fractions; undefined values stay
empty. No final figure files are exported.

In [ ]:
outputs = {"census_agreement_counts.csv": agreement_counts,
           "census_irr.csv": irr,
           "census_descriptive_counts.csv": descriptive_counts,
           "census_taxonomy_counts.csv": taxonomy_counts}
input_locations = {p.resolve() for p in input_paths}
for name in outputs:
    if (OUTPUT_DIR / name).resolve() in input_locations:
        raise ValueError("An output path must not overwrite an input.")
display(pd.DataFrame([{"output": name, "rows": len(frame)} for name, frame in outputs.items()]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, frame in outputs.items():
    exported = frame.assign(input_signature=input_signature)
    exported.to_csv(OUTPUT_DIR / name, index=False, na_rep="")
    print(f"Saved {name}: {len(exported)} rows")

## AI definition and references

The study's operational definition was based on the FDA's understanding of AI, encompassing
machine-learning models and knowledge-based inference systems. Classification required
supporting evidence in FDA summaries or manufacturer websites; automation, prediction or
parameter adaptation alone was insufficient. These evidence criteria operationalize the
FDA concept for this study. Existing AI and AI explicitly introduced through a PCCP both
map to AI in this binary analysis; timing is a separate characteristic and is not inferred
from these verdicts.

1. U.S. Food and Drug Administration. *Marketing Submission Recommendations for a Predetermined
   Change Control Plan for Artificial Intelligence-Enabled Device Software Functions.* Final
   guidance, 18 August 2025, section IV.A, printed p. 8.
   [FDA guidance](https://www.fda.gov/media/166704/download#page=12).
2. U.S. Food and Drug Administration. *Proposed Regulatory Framework for Modifications to
   Artificial Intelligence/Machine Learning (AI/ML)-Based Software as a Medical Device (SaMD):
   Discussion Paper and Request for Feedback.* 2019, section II, p. 4. This discussion paper
   explicitly includes rule-based expert systems.
   [FDA discussion paper](https://www.fda.gov/media/122535/download#page=4).
